# LLM 2일차 (2) — LangChain 챗봇 : `MessagesPlaceholder` 로 대화 기억하기

수업 필기 `LangChain_실습3` 을 정리 · 개선한 노트입니다. (교재 `AI Agent` 교재10 **p.25~26** — 챗봇 구현 실습)

```
LLM 은 지난 대화를 기억하지 못한다 → 지금까지의 대화(chat_history)를 매번 통째로 다시 보낸다
[system] + [Human 1, AI 1, Human 2, AI 2, …, Human n]  →  LLM  →  AI n
```

> **출력 읽는 법** — 정리 노트북은 **API 키 없이** 만들었다(Day 046 과 같은 방식).
>
> | 표시 | 뜻 |
> |---|---|
> | `# 📼` | **수업 시간에 실제로 실행한 결과**를 필기 원본에서 그대로 옮긴 출력 |
> | `# ⏳` | API 키가 필요해 **출력이 비어 있는 셀** — 직접 실행 |
> | `# 🧪` | API 대신 **정해 둔 답을 내는 가짜 LLM** 으로 체인 · 에이전트의 **흐름**만 실제로 돌린 셀 (답 내용은 의미 없음) |
> | 표시 없음 | 정리 노트북에서 실제로 실행한 셀 (프롬프트 채우기 · 토큰 세기 · 도구 정보 · 기록 분석) |

## 0. 설치 · 실행 방법

| 패키지 | 이 노트에서 쓰는 것 | 설치 |
|---|---|---|
| `langchain` · `langchain-openai` | `ChatPromptTemplate` · `MessagesPlaceholder` · `HumanMessage` · `AIMessage` · `trim_messages` | `pip install langchain langchain-openai` |
| `python-dotenv` · `tiktoken` | API 키 · 토큰 세기 | `pip install python-dotenv tiktoken` |

**API 키** : `.env` → `OPENAI_API_KEY="sk-…"`. **실행** : 맨 위부터 모두 실행. 원본의 `while True: input()` 대화는 노트북 자동 실행에서는 멈추므로, 질문 목록을 차례로 보내는 `chat()` 함수로 바꿨다(직접 입력하는 반복문은 맨 끝에 주석으로).

In [1]:
# ① 설치 (처음 한 번)
# !pip install -q langchain langchain-openai python-dotenv tiktoken

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

In [3]:
# .env 파일 → 환경 변수 (키 값은 절대 print 하지 않는다)
import os
from dotenv import load_dotenv, find_dotenv
loaded = load_dotenv(find_dotenv(usecwd=True))      # 지금 폴더에서 위로 올라가며 .env 를 찾는다
print('.env 읽음 :', loaded)
print('OPENAI_API_KEY 있음 :', bool(os.getenv('OPENAI_API_KEY')), '| TAVILY_API_KEY 있음 :', bool(os.getenv('TAVILY_API_KEY')))

.env 읽음 : False
OPENAI_API_KEY 있음 : False | TAVILY_API_KEY 있음 : False


## 1. 대화형 프롬프트 — `MessagesPlaceholder`

In [4]:
prompt = ChatPromptTemplate.from_messages([
    ('system', '너는 친절하고 전문적인 AI 비서야. 이전 대화 맥락을 정확하게 기억하여 사용자의 질문에 일관성 있게 답변해야 해'),
    MessagesPlaceholder(variable_name='chat_history'),     # 대화 기록 리스트가 통째로 들어갈 자리
])
print('채울 변수 :', prompt.input_variables)

history = [HumanMessage('AI 서비스 기획에서 먼저 정할 3가지는?'), AIMessage('① 문제 ② 고객 ③ MVP 범위입니다.'),
           HumanMessage('두 번째를 자세히 알려줘')]
for m in prompt.format_messages(chat_history=history):
    print(f'{type(m).__name__:14s} {m.content}')

채울 변수 : ['chat_history']
SystemMessage  너는 친절하고 전문적인 AI 비서야. 이전 대화 맥락을 정확하게 기억하여 사용자의 질문에 일관성 있게 답변해야 해
HumanMessage   AI 서비스 기획에서 먼저 정할 3가지는?
AIMessage      ① 문제 ② 고객 ③ MVP 범위입니다.
HumanMessage   두 번째를 자세히 알려줘


`'두 번째'` 만으로는 무엇인지 모르지만, 앞의 대화가 함께 들어가 있어서 LLM 이 '고객' 이라는 것을 안다 — 이것이 '기억' 의 정체다.

> **📝 고친 부분 (원본 cell 7 · 10)** `print(prompt)` · `print(chatbot_chain)` 이 수천 글자의 타입 정보(`typing.Annotated[…]`)를 출력 → 위처럼 `format_messages()` 로 실제 전송 내용을 본다. 쓰지 않는 `import os` 삭제.

## 2. 체인 · 대화 (수업 결과)

In [ ]:
# 📼 아래 출력 = 수업 시간에 실행한 결과 (필기 원본에서 옮김)

llm = ChatOpenAI(model='gpt-5-nano')
chatbot_chain = prompt | llm | StrOutputParser()

chat_history = []
print('대화형 AI 챗봇이 시작되었습니다!. \n')
while True:
    user_input = input('사용자 입력: ')
    if user_input.lower() in ['q', 'quit', 'exit', '종료']:
        break
    chat_history.append(HumanMessage(content=user_input))         # 사용자 질문을 기록에 추가
    response = chatbot_chain.invoke({'chat_history': chat_history})
    print(f'AI 챗봇 답변: {response}\n')
    chat_history.append(AIMessage(content=response))              # AI 답변도 기록에 추가

대화형 AI 챗봇이 시작되었습니다!. 

AI 챗봇 답변: 아래 세 가지를 초기 기획에서 가장 먼저 결정하면 좋아요.

1) 해결하려는 문제와 가치 제안(ROI) 정의
- 어떤 소상공인 문제를 해결하고 싶은지 명확히 하고, 기대 효과를 수치(KPI)로 설정합니다(예: 매출 증가, 비용 절감, 작업 시간 감소).

2) 대상 고객(세그먼트)과 핵심 사용 시나리오
- 타깃 업종/기업 규모/기술 수용도 등의 구체적 세그먼트를 정의하고, 가장 가치 있는 2–3가지 사용 사례와 해당 흐름을 명시합니다.

3) 데이터/기술 가능성 및 MVP 범위
- 필요한 데이터 소스, 접근성, 프라이버시/보안 이슈 파악. 기술 선택(모델 유형, 인프라)과 함께, 최소한의 기능으로 가치를 검증할 수 있는 MVP 범위를 정의합니다. 

필요하면 산업군이나 목표 예산을 알려주면, 이 세 가지를 바탕으로 맞춤형 방향도 같이 정리해 드릴게요.

AI 챗봇 답변: 좋습니다. 두 번째 요소인 대상 고객(세그먼트)와 핵심 시나리오를 실제로 구현하려면 어떤 기술 스택으로 접근하면 효율적인지 정리해 드릴게요. 초점은 세그먼트/시나리오를 정의하고 이를 관리하고 시각화하는 데 필요한 도구와 아키텍처입니다.

1) 노코드/저코드로 빠르게 시도하는 경로 (초기 실험용, 예산 적은 경우)
- 데이터 저장 및 관리
  - Airtable: Segment(세그먼트), Persona(페르소나), UseCase(핵심 시나리오), JourneyStep(여정 단계) 등을 하나의 데이터베이스처럼 관리
- UI/포털 구성
  - Softr, Stacker, Pory 등: Airtable 데이터를 바탕으로 내부 구성원용 포털이나 간단한 관리 화면 구축
- 문서화/정의 도구
  - Notion이나 Coda: 시나리오 맵, 의사결정 로그, KPI 정의를 중앙화
- 설계 도구
  - Figma: 페르소나와 여정 흐름, 와이어프레임 제작
- 자동화/연동
  - Zapier나 Make(과거 Integromat): Airtable과 No

수업 결과에는 `input()` 으로 입력한 **질문 글자가 남지 않았다**(노트북은 답만 기록). 답 내용으로 보면 ① AI 서비스 기획에서 먼저 정할 것 → ② 그중 두 번째(대상 고객 · 시나리오)를 구현할 기술 스택 → ③ 지금까지를 세 문장으로 요약 — 앞 대화를 기억해야 답할 수 있는 질문들이다.

## 3. 기록이 쌓이면 — 매번 보내는 토큰이 늘어난다

In [6]:
import re, tiktoken
enc = tiktoken.get_encoding('o200k_base')
recorded = '''대화형 AI 챗봇이 시작되었습니다!. 

AI 챗봇 답변: 아래 세 가지를 초기 기획에서 가장 먼저 결정하면 좋아요.

1) 해결하려는 문제와 가치 제안(ROI) 정의
- 어떤 소상공인 문제를 해결하고 싶은지 명확히 하고, 기대 효과를 수치(KPI)로 설정합니다(예: 매출 증가, 비용 절감, 작업 시간 감소).

2) 대상 고객(세그먼트)과 핵심 사용 시나리오
- 타깃 업종/기업 규모/기술 수용도 등의 구체적 세그먼트를 정의하고, 가장 가치 있는 2–3가지 사용 사례와 해당 흐름을 명시합니다.

3) 데이터/기술 가능성 및 MVP 범위
- 필요한 데이터 소스, 접근성, 프라이버시/보안 이슈 파악. 기술 선택(모델 유형, 인프라)과 함께, 최소한의 기능으로 가치를 검증할 수 있는 MVP 범위를 정의합니다. 

필요하면 산업군이나 목표 예산을 알려주면, 이 세 가지를 바탕으로 맞춤형 방향도 같이 정리해 드릴게요.

AI 챗봇 답변: 좋습니다. 두 번째 요소인 대상 고객(세그먼트)와 핵심 시나리오를 실제로 구현하려면 어떤 기술 스택으로 접근하면 효율적인지 정리해 드릴게요. 초점은 세그먼트/시나리오를 정의하고 이를 관리하고 시각화하는 데 필요한 도구와 아키텍처입니다.

1) 노코드/저코드로 빠르게 시도하는 경로 (초기 실험용, 예산 적은 경우)
- 데이터 저장 및 관리
  - Airtable: Segment(세그먼트), Persona(페르소나), UseCase(핵심 시나리오), JourneyStep(여정 단계) 등을 하나의 데이터베이스처럼 관리
- UI/포털 구성
  - Softr, Stacker, Pory 등: Airtable 데이터를 바탕으로 내부 구성원용 포털이나 간단한 관리 화면 구축
- 문서화/정의 도구
  - Notion이나 Coda: 시나리오 맵, 의사결정 로그, KPI 정의를 중앙화
- 설계 도구
  - Figma: 페르소나와 여정 흐름, 와이어프레임 제작
- 자동화/연동
  - Zapier나 Make(과거 Integromat): Airtable과 Notion 간 데이터 동기화, 알림, 간단한 워크플로우
- 언제 추천하나요?
  - 팀 규모가 작고, 시장 검증이 급하고, 기술 인프라에 자원을 많이 들이고 싶지 않을 때 optimal

2) MVP용 코드 기반 경로 (확장성 있고, 사용자 수 증가에 대비)
- 프런트엔드
  - Next.js + React + TypeScript
  - 역할: 세그먼트/페르소나 관리 화면, 핵심 시나리오 설계 화면, KPI 대시보드
- 백엔드/서비스 계층
  - FastAPI (Python) 또는 NestJS (Node.js)
  - 역할: 세그먼트/시나리오 CRUD, 관계 맵핑, KPI 계산 로직, 간단한 추천 로직
- 데이터 저장
  - PostgreSQL (관계형 DB)
  - 필요 시 Redis로 임시 캐시/세션 저장
- 인증/보안
  - Firebase Authentication 또는 Auth0
- 데이터 분석/시나리오 로직
  - Python 기반 마이크로서비스로 세그먼트 생성 로직, 간단한 규칙 기반 추천, 필요 시 scikit-learn으로 간단한 클러스터링
- API/연계
  - REST 또는 GraphQL API 설계
  - Webhook/외부 시스템 연동을 위한 인터페이스(API Gateway 형태)
- 인프라/배포
  - Docker로 컨테이너화
  - 클라우드: AWS/GCP/Azure 중 선호하는 한 곳
  - 프런트는 Vercel 또는 Netlify로 배포, 백엔드는 ECS/EKS, Cloud Run, 또는 가볍게 App Engine 등으로 운영
- 관찰성/품질
  - Sentry(오류 모니터링), Prometheus/Grafana(성능 모니터링)
  - 로깅: Elastic Stack 또는 Cloud Logging
- 분석 시각화
  - 프런트에서 Chart.js/D3로 세그먼트별 KPI와 사용 흐름 시각화
- 데이터 관리/거버넌스
  - 간단한 데이터 모델링 도구: Prisma ORM 또는 SQLAlchemy를 통해 모델 관리
- 언제 추천하나요?
  - 팀에 개발 역량이 있고, 시장 반응이 빠르게 확인되어 MVP를 제품화할 의향이 있을 때

3) 선택 시 고려할 핵심 트레이드오프
- 개발 속도 vs 유연성
  - 노코드 경로는 빠르지만 구조적 확장이 어렵고, 복잡한 로직이나 커스텀 UI를 구현하기 어렵습니다.
  - 코드 기반 경로는 초기 개발 시간이 더 걸리지만, 이후 기능 확장성과 데이터 관리 측면에서 유리합니다.
- 비용
  - 노코드 도구는 월 사용료가 계속 발생하지만 초기 비용은 낮습니다.
  - 코드 기반은 초기 비용이 더 들 수 있지만, 대규모 사용자나 데이터 처리에 더 적합합니다.
- 데이터 관리의 복잡도
  - 세그먼트/시나리오가 단순하면 노코드로 충분하지만, 복잡한 규칙, 대규모 데이터 파이프라인이 필요하면 코드 기반이 낫습니다.

맞춤 제안 드리려면 몇 가지를 알려주시면 더 구체적인 스택과 로드맵을 드릴 수 있어요
- 목표 예산과 팀 규모(인원수, 개발 경험)
- 선호하는 개발 방식(노코드 우선 vs 코드 우선)
- 초기 시나리오의 예시 2–3개와 목표 세그먼트(예: 소매점 자영업, 식당, 미용업 등)
- 데이터 소스 유무 및 데이터 수집 방법(설문, 이벤트 로그, CRM 데이터 등)

원하시면 위 정보를 바탕으로 맞춤형 2가지 구체적 스택 구성안과 MVP 로드맵을 바로 제시해 드릴게요.

AI 챗봇 답변: 소상공인 문제 해결과 ROI를 명확히 정의하고 KPI를 설정하며, 대상 고객 세그먼트와 2–3개의 핵심 시나리오를 확정한다.
세그먼트/시나리오 구현에 집중하고, 노코드로 빠르게 검증하거나 코드 기반 아키텍처를 선택해 데이터 소스와 보안/프라이버시를 함께 설계한다.
MVP의 성공 기준은 실사용 가치 검증과 KPI 달성으로, 예산·팀 역량에 맞춘 구체적 스택 선택과 일정 로드맵을 수립한다.

'''
answers = [a.strip() for a in re.split(r'AI 챗봇 답변: ', recorded)[1:]]
print('수업 답변 수 :', len(answers), '| 답변별 토큰 :', [len(enc.encode(a)) for a in answers])

system_tokens = len(enc.encode(prompt.messages[0].prompt.template))
q_tokens = 30                                    # 질문 하나 ≈ 30 토큰으로 가정 (질문 글자는 기록에 없음)
sent = 0
for turn, a in enumerate(answers, 1):
    before = system_tokens + q_tokens * turn + sum(len(enc.encode(x)) for x in answers[:turn - 1])
    sent += before
    print(f'{turn}번째 호출 : 보내는 입력 ≈ {before:5,} 토큰 (지난 답 {turn - 1}개 포함)')
print(f'세 번 대화에 보낸 입력 합 ≈ {sent:,} 토큰')

수업 답변 수 : 3 | 답변별 토큰 : [240, 1167, 134]
1번째 호출 : 보내는 입력 ≈    65 토큰 (지난 답 0개 포함)
2번째 호출 : 보내는 입력 ≈   335 토큰 (지난 답 1개 포함)
3번째 호출 : 보내는 입력 ≈ 1,532 토큰 (지난 답 2개 포함)
세 번 대화에 보낸 입력 합 ≈ 1,932 토큰


- 매 호출마다 **지난 대화 전부**를 다시 보내므로 입력 토큰이 대화 길이에 따라 계속 커진다 — 요금과 속도, 그리고 모델의 최대 입력 길이(gpt-5-nano 272,000 토큰) 문제가 된다.
- 해결 : 오래된 대화를 잘라 내거나(`trim_messages`), 요약해서 하나로 줄인다.

In [7]:
from langchain_core.messages import trim_messages, SystemMessage

long_history = []
for a in answers:
    long_history += [HumanMessage('질문 (기록에 없음)'), AIMessage(a)]
long_history.append(HumanMessage('세 문장으로 요약해 줘'))

counter = lambda msgs: sum(len(enc.encode(m.content)) for m in msgs)   # 토큰 세는 함수
trimmed = trim_messages(long_history, max_tokens=800, token_counter=counter,
                        strategy='last', start_on='human', include_system=False)
print('자르기 전 :', len(long_history), '개 ·', counter(long_history), '토큰')
print('자른 후   :', len(trimmed), '개 ·', counter(trimmed), '토큰 → 최근 대화만 남김')
print([type(m).__name__ for m in trimmed])

자르기 전 : 7 개 · 1574 토큰
자른 후   : 3 개 · 151 토큰 → 최근 대화만 남김
['HumanMessage', 'AIMessage', 'HumanMessage']


## 4. `input()` 대신 함수로 — 그대로 실행되는 챗봇

In [8]:
# 🧪 가짜 LLM : 정해 둔 메시지를 차례로 돌려준다 (API · 인터넷 · 요금 없음)
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage

class ScriptedLLM(GenericFakeChatModel):
    """create_agent 도 쓸 수 있게 bind_tools 만 흉내 낸 가짜 모델"""
    def bind_tools(self, tools, **kwargs):
        return self

def fake_llm(*replies):
    return ScriptedLLM(messages=iter([r if isinstance(r, AIMessage) else AIMessage(content=r) for r in replies]))

In [9]:
def make_chatbot(llm):
    chain = prompt | llm | StrOutputParser()
    history = []
    def chat(user_input):
        history.append(HumanMessage(user_input))
        answer = chain.invoke({'chat_history': history})
        history.append(AIMessage(answer))
        return answer
    return chat, history

In [10]:
# 🧪 모의 실행 — API 없이 정해 둔 답을 내는 가짜 LLM 으로 '흐름' 만 확인

chat, history = make_chatbot(fake_llm('① 문제 ② 고객 ③ MVP 범위를 먼저 정하세요.', '두 번째는 대상 고객입니다. …', '요약 : …'))
for q in ['AI 서비스 기획에서 먼저 정할 3가지는?', '두 번째를 구현하려면?', '세 문장으로 요약해 줘']:
    print(f'사용자 : {q}\nAI     : {chat(q)}')
print('\n기록에 쌓인 메시지 :', len(history), '개 →', [type(m).__name__ for m in history])

사용자 : AI 서비스 기획에서 먼저 정할 3가지는?
AI     : ① 문제 ② 고객 ③ MVP 범위를 먼저 정하세요.
사용자 : 두 번째를 구현하려면?
AI     : 두 번째는 대상 고객입니다. …
사용자 : 세 문장으로 요약해 줘
AI     : 요약 : …

기록에 쌓인 메시지 : 6 개 → ['HumanMessage', 'AIMessage', 'HumanMessage', 'AIMessage', 'HumanMessage', 'AIMessage']


In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)

chat, history = make_chatbot(ChatOpenAI(model='gpt-5-nano'))
for q in ['AI 서비스 기획에서 먼저 정할 3가지는?', '그중 두 번째를 구현하려면 어떤 기술 스택이 좋아?', '지금까지 내용을 세 문장으로 요약해 줘']:
    print(f'사용자 : {q}\nAI     : {chat(q)}\n')

# 직접 입력하며 대화하려면 (Colab · Jupyter 에서 직접 실행)
# while (q := input('사용자 입력: ')).lower() not in ['q', 'quit', 'exit', '종료']:
#     print('AI 챗봇 답변:', chat(q), '\n')

> **📝 고친 부분 (원본 cell 12)**
> - `while True: input()` 반복문은 **노트북을 '모두 실행' 하면 입력을 기다리며 멈추고**, 수업 기록처럼 질문이 남지 않는다 → 질문을 인자로 받는 `chat()` 함수로 바꾸고, 질문과 답을 함께 출력
> - 대화 기록이 끝없이 쌓인다 → `trim_messages` 로 최근 대화만 남기는 방법 추가

## 정리

| 할 일 | 코드 |
|---|---|
| 대화 기록 자리 | `MessagesPlaceholder(variable_name='chat_history')` |
| 기록에 추가 | `history.append(HumanMessage(질문))` → 실행 → `history.append(AIMessage(답))` |
| 실행 | `chain.invoke({'chat_history': history})` |
| 기록 줄이기 | `trim_messages(history, max_tokens=…, token_counter=…, strategy='last')` |